# <span style="color:#ff8c00">03 · Image-encoder benchmark</span>

Sequential, combination-first benchmark of 54 frozen ImageNet CNN backbones crossed with five classical question embeddings (GloVe, fastText, word2vec, BERT-base, BERT-large) and two fusion heads (`product`, `attention`), scored on the full `val2014` split with the 1,000-answer list.

Modes (`MODE` below, in run order): `funnel` (screen every candidate with GloVe and BERT-base on the product head) · `survivors` (run the remaining embeddings on the top third of screened encoders) · `attention` (run the attention head on the top encoders and wildcards) · `report` (leaderboards, and bootstrap intervals).

In [ ]:
MODE = "funnel"       # funnel | survivors | attention | report
ONLY = []             # explicit list of encoder keys; empty = every classical image encoder
LIMIT_ENCODERS = None # smoke tests: run only the first N encoders
LIMIT = None          # smoke tests (question subsample)
SCREEN = ["glove", "bert"]                       # every encoder gets these two question embeddings (product head)
REST = ["fasttext", "word2vec", "bert_large"]    # survivors also get these (product head)
CONTROL = "resnet152.tv_in1k"                    # the random-vector control runs once, during this encoder's screen visit
KEEP_FRAC = 0.34      # survivors = the best third of the screened encoders by their best screen holdout (plus ALWAYS)
TOP_N = 6             # attention: the TOP_N encoders by their best product holdout (over all embeddings)
ALWAYS = ["legacy_xception.tf_in1k"]   # also goes through every step, whatever its rank (the original project backbone; exempt from stopping, recorded as `kept` if it falls behind)
ATTENTION_BERTS = ["bert", "bert_large"]          # every top encoder gets attention with these AND with its own best embedding
WILDCARDS = 3         # the best stopped encoders also get attention with bert (a safety net for "bad on product, good on attention")
EPOCHS = 50
PATIENCE = 4
GPU_STATES = True     # BERT question states on the GPU for both heads (~12-16 GB); False on smaller cards

# <span style="color:#ff8c00">Setup</span>

Paths, the resumable run table (`vqa_lab.benchmark.RunTable`), the encoder list and the data context.

In [ ]:
import pandas as pd, matplotlib.pyplot as plt, vqa_lab
from vqa_lab import benchmark as B
from vqa_lab.config import load_encoders, load_paths
from vqa_lab.data import read_table
from vqa_lab.report import write_report

paths = load_paths()
table = B.RunTable(paths, paths.runs_dir / ("benchmark" + ("_smoke" if LIMIT is not None or LIMIT_ENCODERS is not None else "")))   # smoke flags use a separate table
encoders = load_encoders(paths, "image")
ctx = vqa_lab.context(limit=LIMIT) if MODE != "report" else None
RUN = dict(epochs=EPOCHS, patience=PATIENCE, gpu_states=GPU_STATES)

# <span style="color:#ff8c00">1 · Funnel screen</span>

Screen all candidate image backbones with GloVe and BERT-base (product head).

In [ ]:
if MODE == "funnel":
    keys = [e["key"] for e in encoders]
    candidates = [k for k in ONLY if k in keys] if ONLY else keys
    for key in candidates[:LIMIT_ENCODERS]:
        cells = ([("random", "product")] if key == CONTROL else []) + [(t, "product") for t in SCREEN]   # the control rides on its encoder's visit
        table.visit(ctx, key, cells, MODE, **RUN)

# <span style="color:#ff8c00">2 · Survivors benchmark</span>

Evaluate top third of screened backbones across remaining question embeddings (product head).

In [ ]:
if MODE == "survivors":
    _, untrained = B.screen_scores(table.results, SCREEN)
    if untrained:
        print("untrained:", ", ".join(sorted(untrained)))
    survivors, stopped, kept, cut = B.select_survivors(table.results, SCREEN, REST, KEEP_FRAC, ALWAYS)
    if not survivors:
        print("No screened encoders found.")
    else:
        table.results = B.mark_screen(table.results, SCREEN, stopped, kept, cut)   # labels: done / stopped / kept
        table.save()
        for key in survivors:
            table.visit(ctx, key, [(t, "product") for t in REST], MODE, **RUN)

# <span style="color:#ff8c00">3 · Attention benchmark</span>

Evaluate top backbones on attention fusion with their best embedding and BERT variants, plus wildcards.

In [ ]:
if MODE == "attention":
    plan = B.attention_plan(table.results, SCREEN, TOP_N, ALWAYS, ATTENTION_BERTS, WILDCARDS)
    if not plan:
        print("No product runs found.")
    for key in dict.fromkeys(k for k, _ in plan):   # one visit per encoder: its grid features are extracted once
        table.visit(ctx, key, [(t, "attention") for k, t in plan if k == key], MODE, **RUN)

# <span style="color:#ff8c00">4 · Benchmark report</span>

Screen leaderboard, product pivots over every embedding, attention vs product, head rank correlation, 95 % confidence intervals of the top runs.

The record of this benchmark is `artifacts/runs/benchmark/results.parquet` with `failed.csv`. The intervals read the per-question files in its `scores/` folder, which is only a cache: delete it once the report is done.

The report also merges every finished benchmark into one file, `benchmark_results.md`, next to the notebooks.

In [ ]:
if MODE == "report":
    res = table.results   # the run table (empty on a fresh clone: run the funnel first)
    if res.empty:
        print("No runs yet: run the funnel first.")

if MODE == "report" and len(res):
    trained = B.trained_runs(res)   # a cell that never learned is shown as missing, never compared
    product, attention = trained[trained["fusion"] == "product"], trained[trained["fusion"] == "attention"]

    # 1. Screen leaderboard (GloVe and BERT-base, product head)
    board = B.screen_board(res, SCREEN, table.families)
    print("Screen leaderboard (GloVe & BERT-base, product head):")
    display(board)
    top25 = board["better_val"].dropna().head(25)[::-1]
    fig, ax = plt.subplots(figsize=(8, max(5, len(top25) * 0.3)))
    ax.barh(top25.index, top25.values, color="#ff8c00")
    ax.set_xlabel("VQA accuracy % (val2014, best screen)")
    ax.set_title("Top 25 image encoders (screen: GloVe & BERT-base)")
    plt.tight_layout()
    plt.show()

    # 2. Product head with every embedding (the encoders that went on after the screen)
    order = [t for t in B.CLASSICAL_TEXTS if t in set(product["text"])]
    went_on = product[product["backbone"].isin(res.loc[res["text"].isin(REST), "backbone"])]
    print("\nProduct head val % (surviving encoders):")
    display(went_on.pivot(index="backbone", columns="text", values="val").reindex(columns=order))

    # 3. Attention head, and its gain over the product head on the same encoder and embedding
    if len(attention):
        print("\nAttention head val %:")
        display(attention.pivot(index="backbone", columns="text", values="val"))
        gain = pd.merge(product[["backbone", "text", "val"]], attention[["backbone", "text", "val"]], on=["backbone", "text"], suffixes=("_product", "_attention"))
        gain["gain"] = (gain["val_attention"] - gain["val_product"]).round(2)
        print("\nGain (attention - product) val %:")
        display(gain.sort_values("gain", ascending=False).reset_index(drop=True))

    # 4. Do the encoder rankings of the two heads agree? (Spearman, BERT-base)
    both = pd.concat({h: trained[(trained["fusion"] == h) & (trained["text"] == "bert")].set_index("backbone")["val"] for h in ("product", "attention")}, axis=1).dropna()
    if len(both) > 1:
        print(f"\nSpearman rank correlation (BERT product vs attention, n={len(both)}): {both['product'].corr(both['attention'], method='spearman'):.4f}")

    # 5. Is the best run really ahead? 95 % intervals from resampling val2014 images (question sampling only, not seed noise)
    if not LIMIT:
        print("\nTop runs with 95 % image-bootstrap intervals and the paired gap to the best run:")
        display(B.confidence_table(res, table.run_dir / "scores", read_table(paths, "val")["image_id"].to_numpy(), top=10))
        print("\nAll benchmarks merged into:", write_report(paths))